# 05. String & Datetime Accessors (.str & .dt) (5+ Years Interview Guide)
Exhaustive revision guide to vectorized text transformations (.str) and temporal feature extraction (.dt) on raw_transactions.csv.

### Key 5-Year Interview Concepts Covered:
- **String Accessors**: Dedicated cell for `.str.lower()`, `.str.strip()`, `.str.contains()`, `.str.replace()`, and `.str.extract()`.
- **Datetime Accessors**: Dedicated cell for `.dt.year`/`.dt.month`, `.dt.day_name()`, `.dt.floor()`, and `.dt.tz_localize()`/`.dt.tz_convert()`.

This interactive revision guide uses `data/raw_transactions.csv` for all real-world code examples.

In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns
  transaction_id customer_id merchant_id  ...  transaction_date region is_fraud
0       TX110686      C82845       M2697  ...        07-06-2025  North        0
1       TX107170      C85674       M3868  ...        10/05/2025   East        0

[2 rows x 11 columns]


### Lowercasing Strings: `.str.lower()`
**Explanation**: Standardizes card_type and region text columns.

**Syntax**: `df['card_type'].str.lower()`

In [2]:
print('Lowercased Card Types:\n', df['card_type'].str.lower().value_counts())

Lowercased Card Types:
 card_type
amex          3801
mastercard    3768
discover      3723
visa          3708
Name: count, dtype: int64


### Stripping Whitespace: `.str.strip()`
**Explanation**: Removes hidden whitespace padding from strings.

**Syntax**: `df['region'].str.strip()`

In [3]:
print('Stripped Regions:', df['region'].str.strip().unique()[:4])

Stripped Regions: ['North' 'East' 'West' 'South']


### Pattern Matching: `.str.contains()`
**Explanation**: Filters transactions where card_type contains 'Visa' or 'MasterCard'.

**Syntax**: `df['card_type'].str.contains('Visa|MasterCard', regex=True, na=False)`

In [4]:
visa_mc = df[df['card_type'].str.contains('Visa|MasterCard', regex=True, na=False)]
print('Visa or MasterCard Count:', len(visa_mc))

Visa or MasterCard Count: 7476


### Regex String Replacement: `.str.replace()`
**Explanation**: Standardizes transaction IDs by stripping prefix.

**Syntax**: `df['transaction_id'].str.replace('TX_', '', regex=False)`

In [5]:
numeric_tx_ids = df['transaction_id'].str.replace('TX_', '', regex=False)
print('Stripped Numeric IDs Head:\n', numeric_tx_ids.head())

Stripped Numeric IDs Head:
 0    TX110686
1    TX107170
2    TX108328
3    TX108563
4    TX107002
Name: transaction_id, dtype: object


### Regex Extraction: `.str.extract()`
**Explanation**: Extracts prefix and numerical ID components from customer IDs.

**Syntax**: `df['customer_id'].str.extract(r'(?P<Prefix>[A-Z]+)_(?P<Num>\d+)')`

In [6]:
extracted_ids = df['customer_id'].str.extract(r'(?P<Prefix>[A-Z]+)_(?P<Num>\d+)')
print('Extracted ID Components Head:\n', extracted_ids.head())

Extracted ID Components Head:
   Prefix  Num
0    NaN  NaN
1    NaN  NaN
2    NaN  NaN
3    NaN  NaN
4    NaN  NaN


### Temporal Parts Extraction: `.dt.year` & `.dt.month`
**Explanation**: Extracts calendar year and month numbers from transaction dates.

**Syntax**: `clean_dates.dt.year` / `clean_dates.dt.month`

In [7]:
clean_dates = pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')
print('Transaction Year Breakdown:\n', clean_dates.dt.year.value_counts(dropna=False))

Transaction Year Breakdown:
 transaction_date
2025    10892
2026     4108
Name: count, dtype: int64


### Day Name Extraction: `.dt.day_name()`
**Explanation**: Extracts weekday names to analyze weekly transaction volume patterns.

**Syntax**: `clean_dates.dt.day_name()`

In [8]:
print('Transactions per Day of Week:\n', clean_dates.dt.day_name().value_counts())

Transactions per Day of Week:
 transaction_date
Saturday     2232
Wednesday    2210
Friday       2156
Thursday     2147
Monday       2121
Sunday       2091
Tuesday      2043
Name: count, dtype: int64


### Timestamp Truncation: `.dt.floor()`
**Explanation**: Truncates timestamps to day-level granularity.

**Syntax**: `clean_dates.dt.floor('D')`

In [9]:
print('Floored Day Timestamps Head:\n', clean_dates.dt.floor('D').head())

Floored Day Timestamps Head:
 0   2025-07-06
1   2025-10-05
2   2025-07-26
3   2025-09-06
4   2026-04-17
Name: transaction_date, dtype: datetime64[ns]


### Timezone Localization & Conversion: `.dt.tz_localize()` & `.dt.tz_convert()`
**Explanation**: Localizes transaction timestamps to UTC and converts to Eastern Standard Time (EST).

**Syntax**: `clean_dates.dt.tz_localize('UTC').dt.tz_convert('America/New_York')`

In [10]:
utc_ts = clean_dates.dropna().head().dt.tz_localize('UTC')
ny_ts = utc_ts.dt.tz_convert('America/New_York')
print('UTC Timestamps:\n', utc_ts)
print('New York Timestamps (EST):\n', ny_ts)

UTC Timestamps:
 0   2025-07-06 00:00:00+00:00
1   2025-10-05 00:00:00+00:00
2   2025-07-26 06:22:30+00:00
3   2025-09-06 00:00:00+00:00
4   2026-04-17 00:00:00+00:00
Name: transaction_date, dtype: datetime64[ns, UTC]
New York Timestamps (EST):
 0   2025-07-05 20:00:00-04:00
1   2025-10-04 20:00:00-04:00
2   2025-07-26 02:22:30-04:00
3   2025-09-05 20:00:00-04:00
4   2026-04-16 20:00:00-04:00
Name: transaction_date, dtype: datetime64[ns, America/New_York]


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Weekend Fraud Spike Analysis
**Explanation**: Determine if the fraud rate is statistically higher on weekends versus weekdays.

**Syntax**: `df.assign(is_weekend=clean_dates.dt.dayofweek >= 5).groupby('is_weekend')['is_fraud'].mean()`

In [11]:
clean_dt = pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')
weekend_fraud = df.assign(is_weekend=clean_dt.dt.dayofweek >= 5).groupby('is_weekend')['is_fraud'].agg(['count', 'mean'])
print('Weekend vs Weekday Fraud Rates:\n', weekend_fraud)

Weekend vs Weekday Fraud Rates:
             count      mean
is_weekend                 
False       10677  0.108364
True         4323  0.099237
